## Module overview

This notebook imports the core components of the project from the `Python/` package.

- `Python.framework`  
  Contains the main experiment pipeline, including stagewise annealing training, checkpoint evaluation and selection, final checkpoint summarization, and the high-level wrapper `simflow_stagewise`.

- `Python.model`  
  Defines the model architecture, including the flow-based latent transport maps, the relaxed spike-and-slab generative model, the variational base distribution, and the `build_flow_vi` constructor.

- `Python.simfun`  
  Provides simulation utilities for generating synthetic regression datasets with sparse ground-truth coefficients.

- `Python.config`  
  Stores configuration dataclasses used throughout the pipeline, including annealing, data splitting, and saving options.

The imports below allow the notebook to call the full training-and-evaluation pipeline while keeping the implementation organized across separate source files.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import Python.framework as fw
import Python.model as md
import Python.simfun as sf
import Python.config as cfg

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Experiment setup and execution

This block defines the full experimental configuration and runs one complete stagewise annealing experiment.

- `schedule_cfg` specifies the optimization strategy, including the temperature schedule, warm-up length, number of annealing stages, stagewise epoch limits, learning-rate decay, evaluation frequency, posterior diagnostic sample sizes, and the hard-support threshold.
- `split_cfg` defines the train/validation/test split used for checkpoint evaluation and final reporting.
- `save_cfg` controls whether outputs such as plots, histories, checkpoint manifests, and summary tables are written to disk.
- `fw.simflow_stagewise(...)` executes the full pipeline: data generation, model construction, stagewise annealing, checkpoint selection, final posterior summarization, and optional result saving.

This block is the main entry point for testing the current framework under a chosen experimental configuration.

In [21]:
schedule_cfg = cfg.StagewiseAnnealConfig(
    tau_start=1,
    tau_end=0.1,
    warmup_epochs=300,
    n_anneal_stages=10,
    min_stage_epochs=100,
    max_stage_epochs=300,
    base_lr=5e-5,
    stage_lr_decay=0.7,
    eval_every=25,
    print_every=100,
    diag_R_train=256,
    diag_R_final=4000,
    support_threshold=0.1,
)

split_cfg = cfg.SplitConfig(
    train_frac=0.6,
    val_frac=0.2,
    test_frac=0.2,
    seed=123,
)

save_cfg = cfg.SaveConfig(
    output_dir="./plot/run_seed123",
    save_plots=True,
    save_history_csv=True,
    save_checkpoint_manifest=True,
    save_final_json=True,
    save_predictions_csv=True,
    save_support_sets_json=True,
)

out1 = fw.simflow_stagewise(
    build_flow_vi=md.build_flow_vi,
    family="gaussian",
    simfun=sf.simfun1,
    seed=123,
    n=180,
    p=100,
    snr=3.0,
    true_prop=0.1,
    K_q=8,
    K_g=8,
    schedule_cfg=schedule_cfg,
    split_cfg=split_cfg,
    save_cfg=cfg.SaveConfig(output_dir=None),
)

===== Simulation info =====
{'n': 180, 'p': 100, 'n_active': 10, 'sigma2': 3.2589944203694663, 'sigma': 1.8052685175257077, 'active_idx': array([14, 17, 20, 32, 35, 43, 45, 62, 66, 95], dtype=int64), 'snr': 3.0}
[stage 00 | epoch 0001] tau=1.0000 lr=5.00e-05 loss=767.312805 logg=5.8049 dlogg=+0.0000 S=100 churn=0.000 ok*
[stage 00 | epoch 0100] tau=1.0000 lr=5.00e-05 loss=562.496582 logg=4.9479 dlogg=-0.5800 S= 82 churn=0.180 ok
[stage 00 | epoch 0200] tau=1.0000 lr=5.00e-05 loss=283.563751 logg=5.3912 dlogg=+0.2965 S= 15 churn=0.756 ok*
[stage 00 | epoch 0300] tau=1.0000 lr=5.00e-05 loss=216.562057 logg=5.3417 dlogg=-1.0967 S=  4 churn=0.429 ok
[stage 01 | epoch 0301] tau=0.7943 lr=5.00e-05 loss=216.637222 logg=5.4074 dlogg=+0.0657 S=  5 churn=0.200 ok*
[stage 01 | epoch 0400] tau=0.7943 lr=5.00e-05 loss=198.477814 logg=5.4120 dlogg=+0.0748 S=  4 churn=0.200 ok
[stage 01 | epoch 0500] tau=0.7943 lr=5.00e-05 loss=191.322464 logg=5.4132 dlogg=+0.4060 S=  4 churn=0.000 ok
[stage 01 | epo